In [ ]:
# install required sentiment and preprocessing libraries
!pip install vaderSentiment transformers torch emoji -q

# import core packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import emoji
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

# load Spotify reviews dataset from CSV file
reviews_df = pd.read_csv("reviews.csv")

# drop columns not needed
reviews_df = reviews_df.drop(['Time_submitted', 'Reply', 'Total_thumbsup'], axis=1)
reviews_df.head()

In [ ]:
# clean emojis from review text to standardize input to sentiment models
reviews_df['Review_cleaned'] = reviews_df['Review'].apply(
    lambda x: emoji.replace_emoji(str(x), '') if pd.notna(x) else "")

# display dataframe to verify cleaned text column was added
reviews_df

In [ ]:
# initialize VADER sentiment analyzer
vader = SentimentIntensityAnalyzer()

# compute VADER compound score for each cleaned review
compounds = [vader.polarity_scores(t)['compound'] for t in reviews_df['Review_cleaned']]

# store compound scores in dataframe
reviews_df['vader_compound'] = compounds

# convert compound scores to predicted 1–5 star ratings
reviews_df['vader_predicted_rating'] = [int(round((c + 1) * 2 + 1)) for c in compounds]

# display dataframe to verify VADER outputs were added
reviews_df

In [ ]:
# create Hugging Face sentiment pipeline using a star-rating BERT model
hf_pipe = pipeline("sentiment-analysis",
                   # use multilingual star-rating model
                   model="nlptown/bert-base-multilingual-uncased-sentiment",
                   # change to GPU
                   device=0, truncation=True, max_length=512)

# replace empty strings with placeholder text to avoid model errors
texts = [t if t.strip() else "neutral" for t in reviews_df['Review_cleaned']]

# initialize list to store model outputs
results = []

# run inference in batches to improve runtime and avoid memory issues
for i in range(0, len(texts), 32):
    # append batch predictions to results list
    results.extend(hf_pipe(texts[i:i+32]))

# extract predicted star number from model label text and store in dataframe
reviews_df['hf_predicted_rating'] = [int(r['label'].split()[0]) for r in results]

# display dataframe to verify Hugging Face predictions were added
reviews_df

# **TO DO: Create Visualization 1**
Suggested ideas:
- reviews length
- amount of pos vs. neg. vs. neu reviews

In [ ]:
# TODO: add your first visualization here

# **TO DO: Create Visualization 2**
Suggested ideas:
- rating distribution vs. actual
- review length vs. rating

In [ ]:
# TODO: add your second visualization here

In [ ]:
# compute VADER exact match accuracy - percentage
vader_exact = (reviews_df['Rating'] == reviews_df['vader_predicted_rating']).mean() * 100

# compute VADER off-by-one accuracy - percent within 1 star
vader_off1 = (abs(reviews_df['Rating'] - reviews_df['vader_predicted_rating']) <= 1).mean() * 100

# compute VADER mean absolute error - in stars
vader_diff = abs(reviews_df['Rating'] - reviews_df['vader_predicted_rating']).mean()

# compute Hugging Face exact match accuracy - percent
hf_exact = (reviews_df['Rating'] == reviews_df['hf_predicted_rating']).mean() * 100

# compute Hugging Face off-by-one accuracy - percent within 1 star
hf_off1 = (abs(reviews_df['Rating'] - reviews_df['hf_predicted_rating']) <= 1).mean() * 100

# compute Hugging Face mean absolute error - in stars
hf_diff = abs(reviews_df['Rating'] - reviews_df['hf_predicted_rating']).mean()

# summarize both models’ performance metrics in a table
pd.DataFrame({
    'Model': ['VADER', 'HuggingFace'],
    'Exact Match %': [vader_exact, hf_exact],
    'Off-by-One %': [vader_off1, hf_off1],
    'Avg Error': [vader_diff, hf_diff]})

In [ ]:
# create grid for model comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# create side-by-side bar chart comparing model accuracy metrics
x = np.arange(2)
axes[0, 0].bar(x - 0.175, [vader_exact, hf_exact], 0.35, label='Exact Match', color='steelblue')
axes[0, 0].bar(x + 0.175, [vader_off1, hf_off1], 0.35, label='Off-by-One', color='lightsteelblue')
axes[0, 0].set_ylabel('Accuracy (%)')
axes[0, 0].set_title('Model Accuracy')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(['VADER', 'HuggingFace'])
axes[0, 0].legend()
axes[0, 0].set_ylim(0, 100)

# confusion matrices - actual vs predicted ratings
sns.heatmap(pd.crosstab(reviews_df['Rating'], reviews_df['vader_predicted_rating']),
            annot=True, fmt='d', cmap='Blues', ax=axes[0, 1])
axes[0, 1].set_title('VADER')
axes[0, 1].set_xlabel('Predicted')
axes[0, 1].set_ylabel('Actual')

sns.heatmap(pd.crosstab(reviews_df['Rating'], reviews_df['hf_predicted_rating']),
            annot=True, fmt='d', cmap='Greens', ax=axes[0, 2])
axes[0, 2].set_title('HuggingFace')
axes[0, 2].set_xlabel('Predicted')
axes[0, 2].set_ylabel('Actual')

# show distribution of rating prediction errors for each model
axes[1, 0].hist(reviews_df['Rating'] - reviews_df['vader_predicted_rating'],
                bins=9, range=(-4.5, 4.5), color='steelblue', edgecolor='black')
axes[1, 0].set_title('VADER Errors')
axes[1, 0].axvline(0, color='red', linestyle='--')

axes[1, 1].hist(reviews_df['Rating'] - reviews_df['hf_predicted_rating'],
                bins=9, range=(-4.5, 4.5), color='green', edgecolor='black')
axes[1, 1].set_title('HuggingFace Errors')
axes[1, 1].axvline(0, color='red', linestyle='--')

# compare actual rating distribution to model-predicted distributions
x = np.arange(1, 6)
actual = [sum(reviews_df['Rating'] == i) for i in x]
vader_pred = [sum(reviews_df['vader_predicted_rating'] == i) for i in x]
hf_pred = [sum(reviews_df['hf_predicted_rating'] == i) for i in x]
axes[1, 2].bar(x - 0.25, actual, 0.25, label='Actual', color='gray')
axes[1, 2].bar(x, vader_pred, 0.25, label='VADER', color='steelblue')
axes[1, 2].bar(x + 0.25, hf_pred, 0.25, label='HuggingFace', color='green')
axes[1, 2].set_title('Rating Distribution')
axes[1, 2].legend()

# display plots
plt.tight_layout()
plt.show()

In [ ]:
# examples where HuggingFace is better

# create subset dataframe with actual and predicted ratings
sample_df = reviews_df[['Review', 'Rating', 'vader_predicted_rating', 'hf_predicted_rating']].copy()

# compute absolute prediction error for VADER
sample_df['vader_error'] = abs(sample_df['Rating'] - sample_df['vader_predicted_rating'])

# compute absolute prediction error for Hugging Face
sample_df['hf_error'] = abs(sample_df['Rating'] - sample_df['hf_predicted_rating'])

# display first 5 reviews where Hugging Face has lower error than VADER
sample_df[sample_df['hf_error'] < sample_df['vader_error']].head(5)

In [ ]:
# examples where VADER is better
sample_df[sample_df['vader_error'] < sample_df['hf_error']].head(5)